# Lab 1: Building a Semantic Search Engine for the Federalist Papers

In this lab you will turn all 85 Federalist Papers into a searchable vector database and query it by **meaning** rather than keywords. Read the *Vector Database Concepts* page first.

By the end you should be able to:

- load, chunk, embed, and store a text corpus in Chroma with useful metadata;
- run semantic queries and combine them with metadata filters;
- explain, with examples from your own queries, where semantic search beats keyword search and where it falls short.

| Part | Activity | Time |
| --- | --- | --- |
| Setup | Install packages, download the embedding model | 10 min |
| 1 | Download and parse the papers | 10 min |
| 2 | Chunk and load into Chroma | 10 min |
| 3 | Semantic queries and filters | 15 min |
| 4 | Semantic vs. keyword comparison | 15 min |

Run each cell in order with **Shift+Enter**.

## Setup

You need Python 3.10 or newer. Chroma runs inside your Python process, so there is no server to start.

Run the next cell once to install the packages. (If you installed them already in a virtual environment, you can skip it.)

In [ ]:
%pip install "chromadb>=1.0" requests numpy

Chroma's default embedding model, `all-MiniLM-L6-v2`, runs locally on your CPU with no API key. It downloads about 80 MB the first time it is used, so trigger that download now.

In [ ]:
from chromadb.utils import embedding_functions

# Chroma's built-in local embedding model (all-MiniLM-L6-v2)
embed = embedding_functions.DefaultEmbeddingFunction()

vec = embed(["The judge's ruling"])[0]
print(len(vec))      # 384 numbers per text
print(vec[:5])       # the first few values

### Try it: compare three sentences

Embed *"The judge's ruling"*, *"The court's decision"*, and *"A recipe for banana bread"*, then compute the cosine similarity of each pair. Which pair is closest? Notice that the first two share no content words.

In [ ]:
import numpy as np

def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

sentences = ["The judge's ruling", "The court's decision", "A recipe for banana bread"]
vecs = embed(sentences)

for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        print(f"{cosine(vecs[i], vecs[j]):.3f}  {sentences[i]!r} vs {sentences[j]!r}")

## Part 1: Download and parse the papers

Project Gutenberg's eBook #1404 contains all 85 papers in one plain-text file. Each paper begins with a line like `FEDERALIST No. 10`; the argument starts after the salutation *"To the People of the State of New York:"* and ends with the signature *"PUBLIUS."*

Authorship comes from a fixed table of the standard scholarly attributions rather than from the text file, which is inconsistent. Twelve papers are **disputed** between Hamilton and Madison; we label them that way and hold them out for Lab 2.

If the download fails (Gutenberg sometimes throttles repeated requests), download `pg1404.txt` from the course site, put it next to this notebook, and the cell will use the local copy instead.

In [ ]:
import os, re, requests

LOCAL_FILE = "pg1404.txt"
URL = "https://www.gutenberg.org/cache/epub/1404/pg1404.txt"

if os.path.exists(LOCAL_FILE):
    with open(LOCAL_FILE, encoding="utf-8") as f:
        raw = f.read()
    print("Loaded local copy")
else:
    raw = requests.get(URL, timeout=30).text
    with open(LOCAL_FILE, "w", encoding="utf-8") as f:   # cache it for next time
        f.write(raw)
    print("Downloaded from Project Gutenberg")

print(f"{len(raw):,} characters")

In [ ]:
# Standard attributions (paper numbers)
JAY      = {2, 3, 4, 5, 64}
MADISON  = {10, 14} | set(range(37, 49))
JOINT    = {18, 19, 20}
DISPUTED = set(range(49, 59)) | {62, 63}

def author_of(n):
    if n in JAY:      return "Jay"
    if n in MADISON:  return "Madison"
    if n in JOINT:    return "Hamilton and Madison"
    if n in DISPUTED: return "Disputed"
    return "Hamilton"

def strip_gutenberg(raw):
    """Remove the Project Gutenberg license header and footer."""
    start = raw.find("*** START OF")
    start = raw.find("\n", start) + 1 if start != -1 else 0
    end = raw.find("*** END OF")
    return raw[start:end if end != -1 else len(raw)]

def split_papers(text):
    """Return {paper_number: {'title': ..., 'text': ...}}."""
    header = re.compile(r"^FEDERALIST No\. (\d+)\s*$", re.MULTILINE)
    matches = list(header.finditer(text))
    papers = {}
    for i, m in enumerate(matches):
        num = int(m.group(1))
        if num in papers:          # the file contains No. 70 twice; keep the first
            continue
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        body = text[m.end():end]
        # The first non-blank line after the header is the paper's title
        title = next((ln.strip() for ln in body.splitlines() if ln.strip()), "")
        # Keep only the argument: after the salutation, before the signature
        salutation = body.find("To the People of the State of New York")
        if salutation != -1:
            body = body[body.find("\n", salutation) + 1:]
        signoff = body.find("PUBLIUS")
        if signoff != -1:
            body = body[:signoff]
        papers[num] = {"title": title, "text": body.strip()}
    return papers

papers = split_papers(strip_gutenberg(raw))
print(len(papers), "papers")                 # expect 85
print(papers[10]["title"])
print(papers[10]["text"][:300])

### Check your work

You should have exactly 85 papers, with Hamilton 51, Madison 14, Jay 5, joint 3, and disputed 12.

In [ ]:
from collections import Counter
Counter(author_of(n) for n in papers)

## Part 2: Chunk and load into Chroma

Each paragraph becomes one record, and long paragraphs are split at sentence boundaries into pieces of at most about 150 words. That keeps every chunk under the model's 256-token limit, so nothing is silently cut off.

In [ ]:
def paragraphs(text, min_words=20):
    paras = re.split(r"\n\s*\n", text)
    paras = [" ".join(p.split()) for p in paras]          # undo hard line wraps
    return [p for p in paras if len(p.split()) >= min_words]

def chunk_paragraph(para, max_words=150):
    sentences = re.split(r"(?<=[.!?])\s+", para)
    chunks, current = [], []
    for s in sentences:
        if current and len(" ".join(current + [s]).split()) > max_words:
            chunks.append(" ".join(current))
            current = []
        current.append(s)
    if current:
        chunks.append(" ".join(current))
    return chunks

ids, docs, metas = [], [], []
for num, paper in sorted(papers.items()):
    chunk_no = 0
    for p_no, para in enumerate(paragraphs(paper["text"])):
        for chunk in chunk_paragraph(para):
            ids.append(f"fed{num:02d}-c{chunk_no:03d}")
            docs.append(chunk)
            metas.append({"paper": num, "author": author_of(num),
                          "title": paper["title"], "paragraph": p_no})
            chunk_no += 1

print(len(docs), "chunks")        # expect 1,920
print(ids[0], metas[0])

Now create a persistent collection that uses **cosine** distance. Chroma's default is squared Euclidean (L2), so we set it explicitly.

We pass only text; Chroma calls the embedding model for us. Expect this to take one to three minutes on a laptop CPU.

In [ ]:
import chromadb

client = chromadb.PersistentClient(path="./federalist_db")

# Start fresh if you re-run this cell. Check explicitly rather than
# catching errors, so any real problem with the delete is reported.
existing = [c.name for c in client.list_collections()]
if "federalist" in existing:
    client.delete_collection("federalist")

collection = client.create_collection(
    name="federalist",
    configuration={"hnsw": {"space": "cosine"}},
)

BATCH = 500
for i in range(0, len(docs), BATCH):
    collection.add(ids=ids[i:i+BATCH],
                   documents=docs[i:i+BATCH],
                   metadatas=metas[i:i+BATCH])
    print(f"  stored {min(i + BATCH, len(docs))} / {len(docs)}")

print(collection.count(), "records stored")

### Check your work

Look at one stored record, including its vector.

In [ ]:
rec = collection.get(ids=["fed10-c000"], include=["documents", "metadatas", "embeddings"])
print(rec["metadatas"][0])
print(rec["documents"][0][:200])
print(len(rec["embeddings"][0]))     # 384

The `federalist_db` folder now holds a SQLite file plus the HNSW index files. Peek inside it: the vector database is ordinary files on disk.

In [ ]:
for root, dirs, files in os.walk("./federalist_db"):
    for f in files:
        path = os.path.join(root, f)
        print(f"{os.path.getsize(path):>12,}  {path}")

## Part 3: Semantic queries and filters

A query embeds your text with the same model, finds the nearest stored vectors, and returns them with their cosine distances (**smaller is closer**). Start with a helper that prints results readably.

In [ ]:
def search(query, n=5, where=None, where_document=None):
    res = collection.query(query_texts=[query], n_results=n,
                           where=where, where_document=where_document,
                           include=["documents", "metadatas", "distances"])
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        print(f"[{dist:.3f}] No. {meta['paper']} ({meta['author']}): {doc[:160]}...\n")

search("the dangers of political factions")

Paper No. 10, Madison's famous essay on faction, should appear near the top.

### Metadata filters

Filters work like a SQL `WHERE` clause. Chroma supports `$eq`, `$ne`, `$gt`, `$gte`, `$lt`, `$lte`, `$in`, `$nin`, and combines conditions with `$and` and `$or`.

In [ ]:
# Only Hamilton
search("the power to raise taxes", where={"author": "Hamilton"})

In [ ]:
# Only the papers on the judiciary (Nos. 78-83)
search("judges holding office during good behavior",
       where={"$and": [{"paper": {"$gte": 78}}, {"paper": {"$lte": 83}}]})

In [ ]:
# Madison or Jay
search("relations with foreign nations",
       where={"author": {"$in": ["Madison", "Jay"]}})

### Try it: your own queries

Write at least five queries of your own, using modern phrasing for 18th-century ideas. Suggestions:

- "checks and balances between branches of government"
- "why a large republic is better than a small one"
- "a standing army in peacetime"
- "the president's power to pardon"
- "how senators should be chosen"

For each, record the top result, its paper number, and its distance.

In [ ]:
# Your queries here
search("checks and balances between branches of government")

### Look at the distances

What is the distance of a very good match? What happens when you search for something the papers never discuss? A vector database **always** returns its nearest neighbors, even when nothing is relevant. Distance is the only signal that the results are poor.

In [ ]:
search("climate change")

In [ ]:
search("social media")

## Part 4: Semantic vs. keyword search

Semantic search wins when the query and the text use different words for the same idea; keyword search wins when the exact word is the point. You will find examples of both.

Here is a minimal keyword search over the same chunks: it ranks chunks by how many distinct query words they contain.

In [ ]:
STOP = {"the", "a", "an", "of", "to", "and", "in", "on", "for", "is", "by", "with", "be", "that", "this"}

def keyword_search(query, n=5):
    terms = {w for w in re.findall(r"[a-z]+", query.lower()) if w not in STOP}
    scored = []
    for doc, meta in zip(docs, metas):
        words = set(re.findall(r"[a-z]+", doc.lower()))
        score = len(terms & words)
        if score:
            scored.append((score, meta, doc))
    scored.sort(key=lambda t: -t[0])
    for score, meta, doc in scored[:n]:
        print(f"[{score} terms] No. {meta['paper']} ({meta['author']}): {doc[:160]}...\n")

keyword_search("the president")

Chroma also has a built-in exact-substring filter, `where_document`. It is case-sensitive and can be combined with a semantic query.

In [ ]:
# Semantic ranking, restricted to chunks that literally contain "Montesquieu"
search("separation of powers", where_document={"$contains": "Montesquieu"})

### Run these comparisons

Run each query with both `search()` and `keyword_search()`.

| Query | What to notice |
| --- | --- |
| "the president" | The authors usually say "the Executive" or "the Chief Magistrate" |
| "the president's power to pardon" | Here the key word does appear in the text |
| "Montesquieu" | A proper name, cited in a handful of papers |
| "the people should not fear a strong national government" | A whole idea, not a keyword |
| "Amphictyonic council" | A rare historical term |

In [ ]:
COMPARISONS = [
    "the president",
    "the president's power to pardon",
    "Montesquieu",
    "the people should not fear a strong national government",
    "Amphictyonic council",
]

for q in COMPARISONS:
    print("=" * 100)
    print("QUERY:", q)
    print("-" * 40, "SEMANTIC", "-" * 40)
    search(q, n=3)
    print("-" * 40, "KEYWORD", "-" * 41)
    keyword_search(q, n=3)

### Try it

Find one query of your own where semantic search clearly beats keyword search, and one where keyword search clearly wins. Save both for your write-up.

Many production systems run both methods and merge the results; this is called **hybrid search**.

In [ ]:
# Semantic search wins:

In [ ]:
# Keyword search wins:

## Deliverables

Submit this notebook plus a short write-up (one page) covering:

1. Your five Part 3 queries, with the top result's paper number and distance for each.
2. One query where semantic search beat keyword search, and one where it lost, with an explanation of why in each case.
3. Answers to the discussion questions below.

### Discussion questions

1. What range of distances did you see for good matches, and for off-topic queries? Could you choose a cutoff that separates them reliably?
2. Keep the `federalist_db` folder, restart the kernel, and reconnect with `PersistentClient`. Is the data still there? What does that tell you about where Chroma stores state?
3. Every chunk stores the paper's title and author, repeated many times. In relational terms, what normal form does this violate, and why might a vector database accept that trade-off?
4. Our chunks are paragraphs of up to 150 words. Predict what would happen to search quality with whole-paper chunks, and with single sentences.

In [ ]:
# Discussion question 2: after restarting the kernel, run this cell on its own
import chromadb
client = chromadb.PersistentClient(path="./federalist_db")
collection = client.get_collection("federalist")
print(collection.count(), "records still stored")